# Python Fundamentals, via the Kidney Allocation Study

Welcome! This notebook teaches core Python and data-analysis skills using the
real data and logic from **Study 3** of this repo (`Study 3 - AAAI 2026/`),
which is now published as:

> *Moral Change or Noise? On Problems of Aligning AI With Temporally Unstable
> Human Feedback*, Keswani, Cousins, Nguyen, Conitzer, Heidari, Schaich Borg,
> Sinnott-Armstrong. AAAI Alignment Track, 2026.

In that study, participants repeatedly chose which of two kidney-transplant
candidates should receive an available kidney, based on patient attributes
(e.g. years on the waiting list, dependents, criminal history). Researchers
asked the *same* comparison more than once, hours or days apart, to see
whether people's choices stayed stable over time.

We'll rebuild, piece by piece, simplified versions of the real functions in
`helpers.py` and `stability_utils.py` — so by the end you'll understand both
Python fundamentals *and* how the paper's analysis works.

**How to use this notebook:** read each markdown section, run the example
code cell(s), then do the exercise cell marked **Your turn**. Solutions are
given directly below each exercise — try first before peeking.

**Setup:** you'll need `pandas`, `numpy`, `matplotlib`, `seaborn`, and
`scikit-learn`. If anything is missing: `pip install pandas numpy matplotlib seaborn scikit-learn`.


## 1. Variables and data types

A single pairwise comparison in this study has two patients, each described
by several attributes. Let's represent one patient's attributes using plain
Python variables first.


In [ ]:
# A patient's "years waiting" for a kidney, and whether they have dependents
years_waiting = 7
has_dependents = 2      # number of dependents, an integer
reject_chance = 10      # percent chance the body rejects the kidney, a float-ish int
patient_id = "p1"       # a string

print(type(years_waiting), type(reject_chance), type(patient_id))


**Your turn:** make three variables describing a *second* patient (`p2`):
`years_waiting_2`, `has_dependents_2`, `reject_chance_2`. Give them different
values than patient 1. Then print whether patient 1 or patient 2 has waited
longer, using a comparison operator (`>`, `<`, `==`).


In [ ]:
# Your turn — write your code here



<details><summary>Solution</summary>

```python
years_waiting_2 = 3
has_dependents_2 = 0
reject_chance_2 = 15

print(years_waiting > years_waiting_2)  # True: patient 1 waited longer
```
</details>


## 2. Lists and dictionaries

Juggling separate variables per patient gets unwieldy fast. The actual study
code (see `repeated_scenarios.py`) represents each patient as a **dictionary**
mapping feature name -> value, and a full comparison ("scenario") as a
dictionary of two patients. This is exactly how the real scenario `"G2D4"`
is defined in the source code:


In [ ]:
scenario_G2D4 = {
    "p1": {"dep": 0, "alco": 0, "crim": 0, "life": 10, "years_waiting": 5,
           "work_hours": 20, "obesity": 3, "reject_chance": 5},
    "p2": {"dep": 3, "alco": 1, "crim": 2, "life": 10, "years_waiting": 5,
           "work_hours": 20, "obesity": 1, "reject_chance": 5},
}

# Access a single feature of a single patient with two lookups:
print(scenario_G2D4["p1"]["years_waiting"])
print(scenario_G2D4["p2"]["crim"])

# The list of feature names used throughout the study:
feats = ["dep", "alco", "crim", "life", "years_waiting", "work_hours", "obesity", "reject_chance"]
print(feats)


**Your turn:** write a dictionary `scenario_mine` with your own `"p1"` and
`"p2"` patients (pick any values for the 8 features in `feats`). Then print
patient 1's `"obesity"` value.


In [ ]:
# Your turn — write your code here



<details><summary>Solution</summary>

```python
scenario_mine = {
    "p1": {"dep": 1, "alco": 0, "crim": 0, "life": 15, "years_waiting": 2,
           "work_hours": 40, "obesity": 1, "reject_chance": 4},
    "p2": {"dep": 0, "alco": 2, "crim": 1, "life": 20, "years_waiting": 6,
           "work_hours": 10, "obesity": 2, "reject_chance": 8},
}
print(scenario_mine["p1"]["obesity"])
```
</details>


## 3. Loops and conditionals

Instead of comparing one feature at a time by hand, we loop over `feats`.
This is the same pattern the real `helpers.py` uses to build "diff" columns
(`dep_diff`, `alco_diff`, ...) for every feature at once.


In [ ]:
p1 = scenario_G2D4["p1"]
p2 = scenario_G2D4["p2"]

for f in feats:
    diff = p1[f] - p2[f]
    if diff > 0:
        print(f"{f}: p1 is higher by {diff}")
    elif diff < 0:
        print(f"{f}: p2 is higher by {-diff}")
    else:
        print(f"{f}: tied")


**Your turn:** write a loop that counts how many of the 8 features are
*tied* between `p1` and `p2` in `scenario_G2D4`, and print that count.


In [ ]:
# Your turn — write your code here



<details><summary>Solution</summary>

```python
tied = 0
for f in feats:
    if p1[f] == p2[f]:
        tied += 1
print(tied)
```
</details>


## 4. Functions

Let's wrap the comparison logic into a reusable **function** — this is how
real analysis code avoids repeating itself. We'll write a simplified version
of a "which patient wins on feature X" check.


In [ ]:
def higher_on_feature(patient_a, patient_b, feature):
    """Return 'a', 'b', or 'tie' depending on who has the higher value."""
    if patient_a[feature] > patient_b[feature]:
        return "a"
    elif patient_a[feature] < patient_b[feature]:
        return "b"
    return "tie"

print(higher_on_feature(p1, p2, "years_waiting"))
print(higher_on_feature(p1, p2, "crim"))


**Your turn:** write a function `count_wins(patient_a, patient_b, feats)`
that returns how many of the features in `feats` patient `a` is strictly
higher on (reuse `higher_on_feature` inside it).


In [ ]:
# Your turn — write your code here



<details><summary>Solution</summary>

```python
def count_wins(patient_a, patient_b, feats):
    wins = 0
    for f in feats:
        if higher_on_feature(patient_a, patient_b, f) == "a":
            wins += 1
    return wins

print(count_wins(p1, p2, feats))
```
</details>


## 5. Pandas basics: loading the real study data

Now let's move from toy dictionaries to the actual dataset collected for the
paper. Every row is one pairwise comparison a participant made: the left
patient's features (`l_*`), the right patient's features (`r_*`), which one
they `chosen` (0 = left, 1 = right), who they are (`tid`), and which session
this was (`session_number`).


In [ ]:
import pandas as pd

# Path is relative to this notebook's folder, pointing at Study 3's data
data_path = "../Study 3 - AAAI 2026/data/main_response_data/main_response_data_qualtrics_participants.csv"
df = pd.read_csv(data_path)

print(df.shape)       # (num_rows, num_columns)
df.head()


In [ ]:
df.columns.tolist()


`attfailed` flags whether a participant failed an attention check. The real
analysis always drops those rows first:


In [ ]:
df = df[df.attfailed == 0].reset_index(drop=True)
print(df.shape)


**Your turn:** how many *unique* participants (`tid`) are in the cleaned
data? (Hint: `.nunique()`)


In [ ]:
# Your turn — write your code here



<details><summary>Solution</summary>

```python
df.tid.nunique()
```
</details>


## 6. Filtering rows and creating new columns

A **boolean mask** (a column of `True`/`False`) lets you filter a dataframe.
Let's look at just one participant, then build the "diff" columns the real
`helpers.py` uses for modeling (feature value for the left patient minus the
right patient, for every feature in `feats`).


In [ ]:
one_user = df[df.tid == df.tid.iloc[0]]
print(len(one_user), "responses from participant", df.tid.iloc[0])

for f in feats:
    df[f + "_diff"] = df["l_" + f] - df["r_" + f]

diff_cols = [f + "_diff" for f in feats]
df[diff_cols].head()


**Your turn:** filter `df` down to only rows from `session_number == 1`, and
check how many rows that leaves.


In [ ]:
# Your turn — write your code here



<details><summary>Solution</summary>

```python
session_1 = df[df.session_number == 1]
len(session_1)
```
</details>


## 7. Groupby and aggregation

`groupby` answers questions like "for each participant / scenario, what's
the average X?" — the core operation behind most of the paper's statistics.


In [ ]:
# Average "chosen" value per scenario (pairid) — close to 0.5 means no strong preference
df.groupby("pairid")["chosen"].mean().sort_values(ascending=False).head(10)


In [ ]:
# Number of sessions each participant completed
sessions_per_user = df.groupby("tid")["session_number"].nunique()
sessions_per_user.describe()


**Your turn:** compute the number of responses (rows) per participant
(`tid`), and print the 5 participants with the *most* responses.


In [ ]:
# Your turn — write your code here



<details><summary>Solution</summary>

```python
responses_per_user = df.groupby("tid").size().sort_values(ascending=False)
responses_per_user.head(5)
```
</details>


## 8. Putting it together: measuring response stability

This is the heart of the paper. Some scenarios (e.g. `"G2D4"`) were shown
**twice** to the same participant — once as `"G2D4"`, and again, patients
swapped, as `"G2D4_r"`. If a participant is perfectly stable, picking the
same patient both times means their `chosen` answers should be exact
opposites (since the patients swapped sides).

The real function lives in `stability_utils.py` as `get_stability`. Here is
a simplified version for the `"stability"` method: given a list of 0/1
responses to the *same* repeated scenario, how consistent was the
participant?


In [ ]:
import numpy as np

def get_stability(responses):
    """responses: list of 0/1 choices to repeats of the same scenario.
    Returns a value in [0.5, 1]: 1 means always picked the same side, 0.5 means a coin flip."""
    mean_response = np.mean(responses)
    return max(mean_response, 1 - mean_response)

print(get_stability([1, 1, 1]))   # fully stable -> 1.0
print(get_stability([1, 0, 1]))   # one disagreement
print(get_stability([1, 0]))      # perfectly split -> 0.5


**Your turn:** pick any participant's `tid` from `df`, filter to rows where
`pairid == "G2D4"` for that participant, pull out the `chosen` column as a
list, and compute `get_stability` on it. (It's fine if some participants
only saw the scenario once — handle that by checking `len(...) > 0` first.)


In [ ]:
# Your turn — write your code here



<details><summary>Solution</summary>

```python
some_tid = df.tid.iloc[0]
rows = df[(df.tid == some_tid) & (df.pairid == "G2D4")]
responses = rows["chosen"].tolist()
if len(responses) > 0:
    print(responses, get_stability(responses))
else:
    print("This participant didn't see G2D4 — try a different tid.")
```
</details>


## 9. Visualization

A quick bar chart of average `chosen` rate (how often the left patient was
picked) across the top scenarios by number of responses — a much-simplified
cousin of the figures in `aaai_images/`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top_scenarios = df.pairid.value_counts().head(8).index
plot_df = df[df.pairid.isin(top_scenarios)]

plt.figure(figsize=(7, 4))
sns.barplot(data=plot_df, x="pairid", y="chosen")
plt.ylabel("Fraction choosing the left patient")
plt.xlabel("Scenario")
plt.title("Average choice rate for the 8 most-seen scenarios")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Your turn:** make the same kind of bar plot, but for `reaction_time`
instead of `chosen`, for the same `top_scenarios`. Does reaction time vary
much by scenario?


In [ ]:
# Your turn — write your code here



<details><summary>Solution</summary>

```python
plt.figure(figsize=(7, 4))
sns.barplot(data=plot_df, x="pairid", y="reaction_time")
plt.ylabel("Reaction time (seconds)")
plt.xlabel("Scenario")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
```
</details>


## 10. Capstone: a simple preference model

The real paper trains models (logistic regression, random forests, even
neural nets and DPO) to predict a participant's `chosen` answer from the
feature differences between patients — see `get_clf_model_performance` in
`helpers.py`. Let's build the simplest version of that ourselves: a
**logistic regression** predicting `chosen` from `diff_cols`.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df[diff_cols]
y = df["chosen"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))


In [ ]:
# Which features matter most? Bigger absolute coefficient = bigger influence on the choice.
coef_df = pd.Series(model.coef_[0], index=diff_cols).sort_values()
coef_df.plot(kind="barh", figsize=(6, 4), title="Logistic regression coefficients")
plt.tight_layout()
plt.show()


**Your turn:** swap `LogisticRegression` for `sklearn.ensemble.RandomForestClassifier`
(same `X_train`/`y_train`/`X_test`/`y_test`), fit it, and compare its accuracy
to the logistic regression above.


In [ ]:
# Your turn — write your code here



<details><summary>Solution</summary>

```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("Random forest accuracy:", accuracy_score(y_test, rf_pred))
```
</details>


## Where to go next

You've now touched every fundamental used throughout this repo's analysis
code: variables, dicts/lists, loops, conditionals, functions, pandas
filtering/groupby, plotting, and a basic classifier.

To go deeper into the real study:
- `../Study 3 - AAAI 2026/helpers.py` — model fitting, SHAP, DPO/Bradley-Terry
  models, and a simple RL environment (uses PyTorch + stable-baselines3 —
  don't worry about understanding these yet).
- `../Study 3 - AAAI 2026/stability_utils.py` — the real `get_stability` and
  the plotting functions behind the paper's main figures.
- `../Study 3 - AAAI 2026/main_analysis.ipynb` — the full analysis notebook
  that generated every result and figure in the paper.
- `../Study 3 - AAAI 2026/AAAI_Paper.pdf` — the paper itself.

A good next exercise: open `main_analysis.ipynb` and find the cell that
calls `get_stability` — you should now be able to follow exactly what it's
doing.
